# Dehejia & Wahba (1999, *JASA*): Tables 3–4 and covariate balance

Three versions of each result are compared:

| Label | What it is |
|---|---|
| **(i) Paper** | the values printed in the paper |
| **(ii) Reproduction** | our reproduction using the method the paper describes (Table 3 notes and the Appendix) |
| **(iii) SH** | our reproduction using the setup in SH's note (`DW_propensity_score_notes.html`) |

- **Part 1:** Table 3 (estimates) and Table 4 (matched samples), with a covariate-balance check for matching.
- **Part 2:** the covariate-balance check for stratification, showing which covariates fail and in which strata. No adjustments are made here.

Comparison groups: **PSID-1** (`psid_controls`), **CPS-1** (`cps_controls`) and **CPS-3** (the 429 comparison units in `cps3re74`).
Put the notebook in the folder with the `.dta` files. Packages: `numpy`, `scipy`, `pandas`.
**Run time:** a few minutes with 500 bootstrap replications. Set `REPS = 100` for a quick run.

### Reproduction vs. SH

| | (ii) Reproduction (the paper's method) | (iii) SH |
|---|---|---|
| Score | logit with a constant; covariates as in Table 3 notes (e) and (g), **with `nodegree`** | logit with a constant; the same covariates **without `nodegree`** |
| Strata for (3)–(6) | Appendix algorithm: 5 equal score ranges on the common support; split a stratum in half when the **score or any covariate** differs significantly ($|t|>1.96$), provided each half keeps ≥ 3 treated and ≥ 3 comparison units | 5 equal-width blocks on [0, 1]; split a block in half while the **mean score** differs between treated and comparison units ($|t|>1.96$) |
| Covariates in (2), (5), (8) | footnote (a): age, age², education, no degree, black, Hispanic, RE74, RE75 | the same |
| Matching (7)–(8) | nearest neighbor with replacement; ties go to the first comparison unit in data order | the same |

Both setups keep all treated units, plus the comparison units whose score lies between the treated minimum and maximum (note c).

**Our choices where neither source is specific:**
- **Merging strata:** a stratum with treated units but no comparison units is merged into its neighbor.
- **Standard errors:** stratification SEs are analytic within-stratum SEs. Matching SEs come from a bootstrap that re-estimates the score and redoes the matching in each replication. JASA does not say how its SEs were computed.
- **Column (8) under SH:** SH's covariates (footnote a) give 1,565 for CPS-1. The paper's 1,616 is reproduced exactly only with the linear set: age, education, black, Hispanic, no degree, married, RE74, RE75, u74, u75.

## 0. Setup

In [16]:
import numpy as np
import pandas as pd
from scipy.special import expit
import os

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
os.chdir("/Users/kwanpyoko/Dropbox/Notability/Class/E673 - Microeconometrics/topic1_package")
REPS = 500      # bootstrap replications for the matching SEs
SEED = 12345

### Data and helpers

- **Score terms** are written as strings: `"age^2"` is a square, `"education*re74"` is an interaction. `add_terms` creates the column when it is needed.
- **Estimation:** `logit_ps` fits the logit by Newton–Raphson; `ols_se` returns a coefficient and its homoskedastic SE (weights optional).

In [17]:
def load(name):
    d = pd.read_stata(f"{name}.dta")
    d = d.drop(columns=[c for c in ["data_id", "age2"] if c in d])
    d = d.rename(columns={"ed": "education", "hisp": "hispanic", "nodeg": "nodegree"})
    d = d.astype(float)
    d["u74"] = (d.re74 == 0).astype(float)          # zero earnings in 1974
    d["u75"] = (d.re75 == 0).astype(float)
    return d.reset_index(drop=True)

nsw = load("nsw_dw")
treated = nsw[nsw.treat == 1].reset_index(drop=True)
c3 = load("cps3re74")
GROUPS = {"PSID-1": load("psid_controls"),
          "CPS-1":  load("cps_controls"),
          "CPS-3":  c3[c3.treat == 0].reset_index(drop=True)}

def add_terms(d, terms):
    for t in terms:
        if t in d:
            continue
        if t.endswith("^2"):
            d[t] = d[t[:-2]] ** 2
        elif t.endswith("^3"):
            d[t] = d[t[:-2]] ** 3
        else:
            a, b = t.split("*")
            d[t] = d[a] * d[b]
    return d

def logit_ps(d, xs):
    X = d[xs].to_numpy(float)
    sd = X.std(0); sd[sd == 0] = 1
    X = np.column_stack([np.ones(len(X)), (X - X.mean(0)) / sd])   # constant included
    y = d.treat.to_numpy()
    b = np.zeros(X.shape[1])
    for _ in range(100):
        p = expit(X @ b)
        step = np.linalg.solve((X * (p * (1 - p))[:, None]).T @ X + 1e-9 * np.eye(len(b)), X.T @ (y - p))
        b += step
        if np.abs(step).max() < 1e-10:
            break
    return expit(X @ b)

def ols_se(d, y, xs, w=None):
    """Coefficient and SE on the first regressor (xs[0]); WLS if w is given."""
    X = d[xs].to_numpy(float).copy()
    sd = X[:, 1:].std(0); sd[sd == 0] = 1
    X[:, 1:] /= sd
    X = np.column_stack([np.ones(len(d)), X])
    w = np.ones(len(d)) if w is None else np.asarray(w, float)
    Y = d[y].to_numpy(float)
    A = (X * w[:, None]).T @ X
    b = np.linalg.lstsq(A, (X * w[:, None]).T @ Y, rcond=None)[0]
    e = Y - X @ b
    s2 = (w * e ** 2).sum() / (len(Y) - np.linalg.matrix_rank(X))
    return b[1], np.sqrt(s2 * np.linalg.pinv(A)[1, 1])

def te(b, se):
    return f"{b:,.0f} ({se:,.0f})"

print("treated:", len(treated), {g: len(c) for g, c in GROUPS.items()})

treated: 185 {'PSID-1': 2490, 'CPS-1': 15992, 'CPS-3': 429}


### Score specifications and covariate lists

In [18]:
# Table 3, notes (e) and (g), in term notation
PAPER_SCORE = {
 "PSID-1": ["age", "age^2", "education", "education^2", "married", "nodegree", "black", "hispanic",
            "re74", "re75", "re74^2", "re75^2", "black*u74"],
 "CPS-1":  ["age", "age^2", "age^3", "education", "education^2", "married", "nodegree", "black",
            "hispanic", "re74", "re75", "u74", "u75", "education*re74"],
}
PAPER_SCORE["CPS-3"] = PAPER_SCORE["CPS-1"]
SH_SCORE = {g: [x for x in xs if x != "nodegree"] for g, xs in PAPER_SCORE.items()}

REG_A = ["age", "age^2", "education", "nodegree", "black", "hispanic", "re74", "re75"]   # footnote (a)
BAL   = ["age", "education", "black", "hispanic", "nodegree", "married", "re74", "re75", "u74", "u75"]

def pool(comp, xs, t=None):
    """Stack treated + comparison units, create the needed terms, attach the score."""
    t = treated if t is None else t
    d = pd.concat([t, comp], ignore_index=True)
    d = add_terms(d, sorted(set(xs + REG_A)))
    d["ps"] = logit_ps(d, xs)
    return d

def support(d):
    pt = d.ps[d.treat == 1]
    return d[(d.treat == 1) | ((d.ps >= pt.min()) & (d.ps <= pt.max()))]

### Strata rules and estimators

- `strata_reproduction` and `strata_sh` are the two splitting rules described above.
- `t_stat` is the two-sample t-statistic used by both rules and by the balance tests.
- `stratify` gives columns (4) and (5).
- `match_nn` gives each comparison unit's weight: the number of times it is used as a nearest neighbor.

In [19]:
def t_stat(g, v):
    a, b = g[g.treat == 1][v], g[g.treat == 0][v]
    if len(a) < 2 or len(b) < 2:
        return 0.0
    se = np.sqrt(a.var() / len(a) + b.var() / len(b))
    return 0.0 if se == 0 else (a.mean() - b.mean()) / se

def in_block(s, a, b):
    return s[(s.ps >= a) & ((s.ps < b) if b < 1 else (s.ps <= b))]

def merge_one_sided(s, blocks):
    """Merge a block with treated but no comparison units into its neighbor."""
    out = []
    for a, b in blocks:
        g = in_block(s, a, b)
        if out and (g.treat == 0).sum() == 0 and (g.treat == 1).sum() > 0:
            out[-1] = (out[-1][0], b)
        else:
            out.append((a, b))
    while len(out) > 1 and (in_block(s, *out[0]).treat == 0).sum() == 0:
        out[:2] = [(out[0][0], out[1][1])]
    return out

def strata_reproduction(s, min_n=3):
    """Appendix rule: split when the score or any covariate differs; halves need >= min_n of each group."""
    lo, hi = s.ps.min(), 1.0
    edges = sorted({lo, hi, *[e for e in (0.2, 0.4, 0.6, 0.8) if lo < e < hi]})
    todo, done = list(zip(edges[:-1], edges[1:])), []
    while todo:
        a, b = todo.pop()
        g = in_block(s, a, b)
        bad = any(abs(t_stat(g, v)) > 1.96 for v in ["ps"] + BAL)
        if bad and (g.treat == 1).sum() >= 2 and (g.treat == 0).sum() >= 2:
            m = (a + b) / 2
            L, R = in_block(s, a, m), in_block(s, m, b)
            if min((L.treat == 1).sum(), (L.treat == 0).sum(), (R.treat == 1).sum(), (R.treat == 0).sum()) >= min_n:
                todo += [(a, m), (m, b)]
                continue
        done.append((a, b))
    return merge_one_sided(s, sorted(x for x in done if len(in_block(s, *x))))

def strata_sh(s):
    """SH's rule: 5 equal-width blocks on [0,1]; split while the mean score differs."""
    todo, done = [(i / 5, (i + 1) / 5) for i in range(5)], []
    while todo:
        a, b = todo.pop()
        g = in_block(s, a, b)
        if (g.treat == 1).sum() >= 2 and (g.treat == 0).sum() >= 2 and abs(t_stat(g, "ps")) > 1.96 and b - a > 1e-6:
            m = (a + b) / 2
            todo += [(a, m), (m, b)]
            continue
        done.append((a, b))
    return merge_one_sided(s, sorted(x for x in done if len(in_block(s, *x))))

def stratify(s, blocks, regx):
    n1 = (s.treat == 1).sum()
    e4 = v4 = e5 = v5 = 0.0
    for a, b in blocks:
        g = in_block(s, a, b)
        t, c = g[g.treat == 1], g[g.treat == 0]
        if len(t) == 0:
            continue
        w = len(t) / n1
        d4 = t.re78.mean() - c.re78.mean()
        var4 = np.nan_to_num(t.re78.var() / len(t)) + np.nan_to_num(c.re78.var() / len(c))
        if len(g) > len(regx) + 3:
            d5, se5 = ols_se(g, "re78", ["treat"] + regx)
        else:                                         # too few units for the regression
            d5, se5 = d4, np.sqrt(var4)
        e4 += w * d4; v4 += w * w * var4
        e5 += w * d5; v5 += w * w * se5 ** 2
    return (e4, np.sqrt(v4)), (e5, np.sqrt(v5))

def match_nn(d):
    t, c = d[d.treat == 1], d[d.treat == 0]
    pc = c.ps.to_numpy()
    w = np.zeros(len(pc))
    for p in t.ps:
        w[np.abs(pc - p).argmin()] += 1                # argmin: first unit in data order on ties
    return w

def matching(d, regx):
    t, c = d[d.treat == 1], d[d.treat == 0]
    w = match_nn(d)
    att = t.re78.mean() - (w @ c.re78.to_numpy()) / len(t)
    u = w > 0
    dm = pd.concat([t, c[u]])
    reg = ols_se(dm, "re78", ["treat"] + regx, np.r_[np.ones(len(t)), w[u]])[0]
    return att, reg, w

def boot_matching(comp, xs, regx, reps=REPS, seed=SEED):
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(reps):
        tb = treated.iloc[rng.integers(0, len(treated), len(treated))]
        cb = comp.iloc[rng.integers(0, len(comp), len(comp))]
        draws.append(matching(pool(cb, xs, tb), regx)[:2])
    return np.std(draws, axis=0, ddof=1)

def table3_row(g, xs, strata_rule, regx=REG_A):
    comp = GROUPS[g]
    d = pool(comp, xs)
    s = support(d)
    blocks = strata_rule(s)
    (e4, s4), (e5, s5) = stratify(s, blocks, regx)
    b3, se3 = ols_se(s.assign(ps2=s.ps ** 2), "re78", ["treat", "ps", "ps2"])
    att, reg, w = matching(d, regx)
    se = boot_matching(comp, xs, regx)
    row = {"(1)": te(*ols_se(d, "re78", ["treat"])), "(2)": te(*ols_se(d, "re78", ["treat"] + REG_A)),
           "(3)": te(b3, se3), "(4)": te(e4, s4), "(5)": te(e5, s5), "(6)": f"{len(s):,}",
           "(7)": te(att, se[0]), "(8)": te(reg, se[1])}
    return row, {"d": d, "s": s, "blocks": blocks, "w": w}

# Part 1. Tables 3 and 4

## Table 3: estimated training effects

For each group: **(i) Paper / (ii) Reproduction / (iii) SH**. The NSW row is the experimental benchmark.

In [20]:
PAPER_T3 = {
 "PSID-1": ["-15,205 (1,154)", "731 (886)", "294 (1,389)", "1,608 (1,571)", "1,494 (1,581)", "1,255", "1,691 (2,209)", "1,473 (809)"],
 "CPS-1":  ["-8,498 (712)", "972 (550)", "1,117 (747)", "1,713 (1,115)", "1,774 (1,152)", "4,117", "1,582 (1,069)", "1,616 (751)"],
 "CPS-3":  ["-635 (657)", "1,326 (798)", "556 (951)", "1,252 (1,617)", "2,219 (2,082)", "514", "587 (1,496)", "662 (776)"],
}
COLS = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)"]
SETUPS = {"(ii) Reproduction": (PAPER_SCORE, strata_reproduction),
          "(iii) SH":          (SH_SCORE,    strata_sh)}

nsw_a = add_terms(nsw.copy(), REG_A)
rows = {("NSW", "(i) Paper"): ["1,794 (633)", "1,672 (638)"] + [""] * 6,
        ("NSW", "(ii) Reproduction"): [te(*ols_se(nsw_a, "re78", ["treat"])),
                                       te(*ols_se(nsw_a, "re78", ["treat"] + REG_A))] + [""] * 6}
results = {}
for g in GROUPS:
    rows[(g, "(i) Paper")] = PAPER_T3[g]
    for label, (scores, rule) in SETUPS.items():
        r, results[(g, label)] = table3_row(g, scores[g], rule)
        rows[(g, label)] = [r[c] for c in COLS]
table3 = pd.DataFrame(rows, index=COLS).T
table3

(1)          (2)          (3)            (4)            (5)    (6)            (7)            (8)
NSW    (i) Paper              1,794 (633)  1,672 (638)                                                                                
       (ii) Reproduction      1,794 (633)  1,672 (638)                                                                                
PSID-1 (i) Paper          -15,205 (1,154)    731 (886)  294 (1,389)  1,608 (1,571)  1,494 (1,581)  1,255  1,691 (2,209)    1,473 (809)
       (ii) Reproduction  -15,205 (1,155)    218 (866)  265 (1,387)    2,154 (790)  2,492 (1,894)  1,331    552 (1,924)    495 (1,590)
       (iii) SH           -15,205 (1,155)    218 (866)  286 (1,388)    1,039 (918)  1,072 (1,336)  1,342  1,691 (1,997)  1,674 (1,316)
CPS-1  (i) Paper             -8,498 (712)    972 (550)  1,117 (747)  1,713 (1,115)  1,774 (1,152)  4,117  1,582 (1,069)    1,616 (751)
       (ii) Reproduction     -8,498 (712)    739 (547)  1,071 (762)    1,564 (733)  1,781 (1,050)  4,041      952 (925)    1,098 (916)
       (iii) SH              -8,498 (712)    739 (547)  1,109 (747)    1,833 (675)  1,877 (1,033)  3,566    1,582 (909)    1,565 (884)
CPS-3  (i) Paper               -635 (657)  1,326 (798)    556 (951)  1,252 (1,617)  2,219 (2,082)    514    587 (1,496)      662 (776)
       (ii) Reproduction       -635 (657)  1,326 (796)    625 (950)    1,587 (884)  1,434 (1,869)    506   -467 (1,183)   -419 (1,100)
       (iii) SH                -635 (657)  1,326 (796)    649 (950)    1,211 (936)  1,541 (1,655)    515    587 (1,152)    777 (1,118)

**Reading Table 3:**
- **Matching, column (7):** SH reproduces the paper exactly for all three groups. Reproduction does not.
- **Column (8):** under SH it is close to the paper, and exact only with the linear covariate set (see above).
- **Stratification, columns (3)–(6):** neither version reproduces the paper. The column (6) sample differs, and the paper's strata are not published.
- **Column (2):** the paper's PSID-1 and CPS-1 values (731, 972) do not follow its own footnote (a), which gives 218 and 739.

## Table 4: matched comparison samples

Means of the matched comparison units, **weighted by the number of times each unit is matched**; that is how the paper's means reproduce. `N` = distinct matched comparison units.

In [21]:
T4 = ["age", "education", "black", "hispanic", "nodegree", "married", "re74", "re75"]
T4_LAB = ["Age", "Education", "Black", "Hispanic", "No degree", "Married", "RE74", "RE75"]
PAPER_T4 = {   # means only, transcribed from Table 4
 "PSID-1": [56, 26.39, 10.62, .86, .02, .55, .15, 1794, 1126],
 "CPS-1":  [119, 26.91, 10.52, .86, .04, .64, .19, 2110, 1396],
 "CPS-3":  [63, 25.94, 10.69, .87, .06, .53, .13, 2709, 1587],
}

def fmt(v, x):
    return f"{x:,.0f}" if v.startswith("re") else f"{x:.2f}"

def matched_means(info):
    d, w = info["d"], info["w"]
    c = d[d.treat == 0]
    u = w > 0
    return [int(u.sum())] + [np.average(c[v][u], weights=w[u]) for v in T4]

rows4 = {("NSW treated", ""): [185] + [fmt(v, treated[v].mean()) for v in T4]}
for g in GROUPS:
    p = PAPER_T4[g]
    rows4[("M" + g, "(i) Paper")] = [p[0]] + [fmt(v, x) for v, x in zip(T4, p[1:])]
    for label in SETUPS:
        m = matched_means(results[(g, label)])
        rows4[("M" + g, label)] = [m[0]] + [fmt(v, x) for v, x in zip(T4, m[1:])]
table4 = pd.DataFrame(rows4, index=["N"] + T4_LAB).T
table4

N    Age Education Black Hispanic No degree Married   RE74   RE75
NSW treated                    185  25.82     10.35  0.84     0.06      0.71    0.19  2,096  1,532
MPSID-1     (i) Paper           56  26.39     10.62  0.86     0.02      0.55    0.15  1,794  1,126
            (ii) Reproduction   55  26.13     10.31  0.92     0.01      0.56    0.16  1,873  1,528
            (iii) SH            52  26.39     10.63  0.86     0.02      0.55    0.15  1,794  1,126
MCPS-1      (i) Paper          119  26.91     10.52  0.86     0.04      0.64    0.19  2,110  1,396
            (ii) Reproduction  106  25.66     10.80  0.88     0.03      0.59    0.18  1,839  1,475
            (iii) SH           112  26.91     10.52  0.86     0.04      0.64    0.19  2,110  1,396
MCPS-3      (i) Paper           63  25.94     10.69  0.87     0.06      0.53    0.13  2,709  1,587
            (ii) Reproduction   63  25.49     10.50  0.89     0.05      0.68    0.16  1,771  1,666
            (iii) SH            64  25.95     10.69  0.87     0.06      0.53    0.13  2,709  1,587

Under SH, every mean equals the paper's to rounding. The paper's `N` (56 / 119 / 63) does not match the number of distinct matched units (52 / 112 / 64).

### Covariate balance for matching

For each covariate, the standardized difference between the NSW treated and the matched comparison sample:

$$d = 100\cdot\frac{\bar x_T-\bar x_{C,\text{matched}}}{\sqrt{(s_T^2+s_C^2)/2}}$$

Here $s_T^2$ and $s_C^2$ are the variances in the full treated and full comparison samples. **Balanced** means $|d|<10$ for every covariate.

- **(i) Paper:** uses the matched means printed in Table 4. Only the 8 covariates in that table are available; u74 and u75 are not reported.
- **(ii) Reproduction and (iii) SH:** use our matched samples, for all 10 covariates.

In [22]:
def std_diff(xbar_c, v, comp):
    return 100 * (treated[v].mean() - xbar_c) / np.sqrt((treated[v].var() + comp[v].var()) / 2)

def matching_balance(g, label):
    comp = GROUPS[g]
    if label == "(i) Paper":
        return {v: std_diff(x, v, comp) for v, x in zip(T4, PAPER_T4[g][1:])}
    info = results[(g, label)]
    d, w = info["d"], info["w"]
    c = d[d.treat == 0]
    u = w > 0
    return {v: std_diff(np.average(c[v][u], weights=w[u]), v, comp) for v in BAL}

LABELS = ["(i) Paper", "(ii) Reproduction", "(iii) SH"]
mbal = pd.DataFrame({(g, lab): matching_balance(g, lab) for g in GROUPS for lab in LABELS}).reindex(BAL)
print("Standardized differences after matching (%); |d| >= 10 is flagged in the summary below")
mbal.round(1)

Standardized differences after matching (%); |d| >= 10 is flagged in the summary below


PSID-1                                CPS-1                                CPS-3                           
          (i) Paper (ii) Reproduction (iii) SH (i) Paper (ii) Reproduction (iii) SH (i) Paper (ii) Reproduction (iii) SH
age            -6.4              -3.5     -6.4     -11.8               1.7    -11.7      -1.4               3.6     -1.4
education     -10.5               1.5    -10.8      -7.0             -18.3     -7.0     -13.9              -6.3    -14.0
black          -4.2             -20.2     -5.4      -5.3             -10.2     -6.8      -7.0             -11.3     -7.0
hispanic       18.8              25.8     20.6       7.8              13.1      8.7      -0.2               1.8     -1.8
nodegree       34.5              31.8     35.4      14.9              24.9     14.2      37.6               5.7     37.6
married        10.7               7.4     10.3      -0.2               1.3     -1.3      13.2               6.0     13.2
re74            3.0               2.2      3.0      -0.2               3.4     -0.2     -10.4               5.5    -10.4
re75            4.1               0.0      4.1       2.0               0.8      2.0      -1.7              -4.1     -1.7
u74             NaN              21.4     14.3       NaN             -15.0     -5.5       NaN               0.0     10.9
u75             NaN               1.3    -18.6       NaN               0.0     -1.3       NaN             -14.7      4.5

In [23]:
summary = {}
for (g, lab), col in mbal.items():
    col = col.dropna()
    bad = col[col.abs() >= 10]
    summary[(g, lab)] = {"max |d|": round(col.abs().max(), 1),
                         "covariates with |d| >= 10": ", ".join(f"{k} ({v:.0f})" for k, v in bad.items()) or "none",
                         "balanced?": "yes" if bad.empty else "no"}
pd.DataFrame(summary).T

max |d|                                                         covariates with |d| >= 10 balanced?
PSID-1 (i) Paper            34.5                       education (-11), hispanic (19), nodegree (35), married (11)        no
       (ii) Reproduction    31.8                               black (-20), hispanic (26), nodegree (32), u74 (21)        no
       (iii) SH             35.4  education (-11), hispanic (21), nodegree (35), married (10), u74 (14), u75 (-19)        no
CPS-1  (i) Paper            14.9                                                          age (-12), nodegree (15)        no
       (ii) Reproduction    24.9             education (-18), black (-10), hispanic (13), nodegree (25), u74 (-15)        no
       (iii) SH             14.2                                                          age (-12), nodegree (14)        no
CPS-3  (i) Paper            37.6                          education (-14), nodegree (38), married (13), re74 (-10)        no
       (ii) Reproduction    14.7                                                            black (-11), u75 (-15)        no
       (iii) SH             37.6                education (-14), nodegree (38), married (13), re74 (-10), u74 (11)        no

**Reading the matching balance:**
- **No version is balanced** for any group, including the paper's own matched samples (computed from its Table 4 means).
- **`nodegree` is the most frequent problem.** It exceeds 10% for PSID-1 and CPS-1 in all three versions, and for CPS-3 under Paper and SH (38%).
- **CPS-1 under Paper and SH:** only `nodegree` (15% and 14%) and age (−12%) fail.
- **PSID-1:** Hispanic fails in all three versions, and u74 in both reproductions. Reproduction also fails on black; Paper and SH also fail on education and married, and SH on u75.
- **CPS-3:** Paper and SH fail on `nodegree`, education, married and RE74 (SH also on u74). Reproduction fails only on black and u75.

# Part 2. Covariate balance for stratification

The Appendix test: within each stratum that has at least 2 treated and 2 comparison units, a two-sample t-test of each covariate's mean, treated vs. comparison. A covariate **fails** in a stratum if $|t|>1.96$.
- **Strictly balanced:** no test fails.
- **Chance level:** with many tests, about 5% fail by chance even when every stratum is balanced. We report that benchmark next to the count.

The paper publishes no strata, so only (ii) Reproduction and (iii) SH can be checked. No adjustment is made here; the goal is to see which covariates cause the problems.

In [24]:
def strata_table(info):
    s, rows = info["s"], []
    for a, b in info["blocks"]:
        blk = in_block(s, a, b)
        n1, n0 = int((blk.treat == 1).sum()), int((blk.treat == 0).sum())
        tested = n1 >= 2 and n0 >= 2
        bad = [v for v in BAL if tested and abs(t_stat(blk, v)) > 1.96]
        rows.append({"score range": f"[{a:.3f}, {b:.3f})", "treated": n1, "comparison": n0,
                     "tested": "yes" if tested else "no (too few units)",
                     "failing covariates": ", ".join(f"{v} (t={t_stat(blk, v):.1f})" for v in bad) or "-"})
    return pd.DataFrame(rows)

def strata_counts(info):
    s, per_var, tests, n_tested = info["s"], {v: 0 for v in BAL}, 0, 0
    for a, b in info["blocks"]:
        blk = in_block(s, a, b)
        if (blk.treat == 1).sum() < 2 or (blk.treat == 0).sum() < 2:
            continue
        n_tested += 1
        for v in BAL:
            tests += 1
            per_var[v] += abs(t_stat(blk, v)) > 1.96
    return per_var, tests, n_tested

sbal, overview = {}, {}
for g in GROUPS:
    for lab in SETUPS:
        info = results[(g, lab)]
        per_var, tests, n_tested = strata_counts(info)
        sbal[(g, lab)] = per_var
        f = sum(per_var.values())
        overview[(g, lab)] = {"strata": len(info["blocks"]), "strata tested": n_tested,
                              "failing tests / tests": f"{f} / {tests}",
                              "expected by chance (5%)": round(0.05 * tests, 1),
                              "result": "strictly balanced" if f == 0 else
                                        ("within chance level" if f <= 0.05 * tests else "NOT balanced")}
pd.DataFrame(overview).T

strata strata tested failing tests / tests expected by chance (5%)        result
PSID-1 (ii) Reproduction      8             8               11 / 80                     4.0  NOT balanced
       (iii) SH               8             8               10 / 80                     4.0  NOT balanced
CPS-1  (ii) Reproduction     18            18              27 / 180                     9.0  NOT balanced
       (iii) SH               9             9                8 / 90                     4.5  NOT balanced
CPS-3  (ii) Reproduction     10            10               8 / 100                     5.0  NOT balanced
       (iii) SH               5             5                6 / 50                     2.5  NOT balanced

### Which covariates cause the problems

Number of strata in which each covariate fails the t-test (out of the strata tested):

In [25]:
problems = pd.DataFrame(sbal).reindex(BAL).astype(int)
problems.loc["total"] = problems.sum()
problems

PSID-1                      CPS-1                      CPS-3         
          (ii) Reproduction (iii) SH (ii) Reproduction (iii) SH (ii) Reproduction (iii) SH
age                       0        0                 3        1                 0        0
education                 0        0                 2        2                 0        1
black                     3        2                 2        0                 2        1
hispanic                  3        2                 3        1                 1        2
nodegree                  0        0                 4        1                 0        0
married                   2        1                 4        1                 1        1
re74                      1        2                 2        1                 3        1
re75                      0        0                 1        0                 0        0
u74                       1        2                 3        0                 1        0
u75                       1        1                 3        1                 0        0
total                    11       10                27        8                 8        6

### Stratum by stratum

In [26]:
for g in GROUPS:
    for lab in SETUPS:
        print(f"{g} — {lab}")
        print(strata_table(results[(g, lab)]).to_string(index=False), "\n")

PSID-1 — (ii) Reproduction
   score range  treated  comparison tested                                                               failing covariates
[0.001, 0.050)        7         918    yes black (t=3.5), hispanic (t=-6.8), married (t=12.7), u74 (t=-11.4), u75 (t=-13.2)
[0.050, 0.100)        4         100    yes                                                                hispanic (t=-2.5)
[0.100, 0.200)        7          53    yes                                                                    re74 (t=-3.5)
[0.200, 0.400)       27          41    yes                                                                                -
[0.400, 0.500)        5           8    yes                                                                   black (t=-2.4)
[0.500, 0.600)       15           6    yes                                                                                -
[0.600, 0.800)       22          13    yes                                                               

**Reading the stratification balance:**
- **No group is balanced under either version.** Failing tests are about 1.6–3 times what chance alone would produce.
- **Covariates that fail most often:** Hispanic, black, married and RE74, in every group. `nodegree`, age and u74/u75 fail repeatedly under Reproduction for CPS-1. u74 and u75 also fail in PSID-1's lowest stratum.
- **Where the failures concentrate:**
  - *The lowest-score strata,* where a few treated units face hundreds or thousands of comparison units. PSID-1's first stratum (7 treated vs. about 920 comparison units) fails on five covariates, and most CPS-1 failures are in strata below a score of 0.25.
  - *The highest-score strata,* where many treated units share a handful of comparison units. For example, PSID-1 above 0.6 under SH has 118 treated vs. 20 comparison units, and CPS-3 above 0.8 has 83 treated vs. 7 comparison units under Reproduction.
- **Next step, if wanted:** the Appendix's remedy is to add interaction or higher-order terms for these covariates and re-estimate the score. That is not done here.